# 03 - Vector Indexing & RAG Knowledge Base (Google Colab)

### Semantic Retrieval Indexing with Sentence-Transformers & Qdrant Cloud
**Project:** Customer Support RAG Chatbot  
**Execution Environment:** Google Colab (CPU or GPU)  
**Target Vector Database:** Qdrant Cloud (Free 1GB Cluster) + Local JSON Fallback  
**Embedding Model:** `BAAI/bge-small-en-v1.5` or `sentence-transformers/all-MiniLM-L6-v2` (384 dimensions)

---
### Objectives
1. Load cleaned customer support problem-solution pairs from Google Drive.
2. Group and extract canonical enterprise knowledge base articles.
3. Generate dense semantic embeddings using Sentence-Transformers.
4. Create the `customer_support_kb` collection in Qdrant with Cosine distance.
5. Batch upsert vectors with rich payload metadata (brand, issue category, resolution).
6. Export an offline `sample_kb.json` (100 representative entries) for local offline testing.



In [ ]:
# Step 1: Install Retrieval Stack
!pip install -q qdrant-client sentence-transformers tqdm



In [ ]:
# Step 2: Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')
DATA_PATH = "/content/drive/MyDrive/chatbot_data/processed/cleaned_customer_support_sample.jsonl"



### Step 3: Load and Categorize Knowledge Base Articles
We extract distinct resolution patterns into structured support articles.



In [ ]:
import json
from tqdm.auto import tqdm

print(f"[*] Reading support pairs from {DATA_PATH}...")
articles = []

with open(DATA_PATH, "r", encoding="utf-8") as f:
    for idx, line in enumerate(f):
        data = json.loads(line.strip())
        inquiry = data["instruction"]
        response = data["response"]
        brand = data["brand"]
        
        # Categorize by simple heuristics
        category = "General Inquiry"
        low_inq = inquiry.lower()
        if any(w in low_inq for w in ["refund", "billing", "charged", "cancel", "payment", "subscription"]):
            category = "Billing & Payments"
        elif any(w in low_inq for w in ["order", "tracking", "delivery", "shipping", "package", "arrived"]):
            category = "Orders & Delivery"
        elif any(w in low_inq for w in ["battery", "update", "screen", "login", "password", "crash", "app", "sync"]):
            category = "Technical Support"
        elif any(w in low_inq for w in ["flight", "booking", "seat", "baggage", "delay", "ticket"]):
            category = "Reservations & Travel"
            
        articles.append({
            "doc_id": idx + 1,
            "brand": brand,
            "category": category,
            "query": inquiry,
            "resolution": response,
            "text": f"Brand: {brand} | Category: {category} | Problem: {inquiry} | Solution: {response}"
        })

print(f"[OK] Structured {len(articles):,} knowledge base articles.")



### Step 4: Initialize Embedding Model
`BAAI/bge-small-en-v1.5` produces high-fidelity 384-dimensional embeddings optimized for retrieval tasks.



In [ ]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"
print(f"[*] Loading embedding model: {EMBEDDING_MODEL_NAME}...")
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)
sample_vec = embedder.encode("Test query")
VECTOR_DIM = len(sample_vec)
print(f"[OK] Embeddings initialized. Vector dimensions: {VECTOR_DIM}")



### Step 5: Connect to Qdrant Cloud (or In-Memory Local)
Provide your Qdrant Cloud cluster URL and API key from secrets or input fields.



In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.http import models

QDRANT_URL = os.getenv("QDRANT_URL", "")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")

# Fallback check from Colab secrets
if not QDRANT_URL:
    try:
        from google.colab import userdata
        QDRANT_URL = userdata.get('QDRANT_URL')
        QDRANT_API_KEY = userdata.get('QDRANT_API_KEY')
    except Exception:
        pass

if QDRANT_URL:
    print(f"[*] Connecting to Qdrant Cloud at: {QDRANT_URL}")
    client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
else:
    print("[!] No Qdrant Cloud URL provided. Initializing local in-memory Qdrant client...")
    client = QdrantClient(":memory:")

COLLECTION_NAME = "customer_support_kb"

# Recreate collection with Cosine Distance
client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=models.VectorParams(
        size=VECTOR_DIM,
        distance=models.Distance.COSINE
    )
)
print(f"[OK] Collection '{COLLECTION_NAME}' created successfully.")



### Step 6: Batch Vector Upsert
Encode knowledge documents and upsert into Qdrant in batches of 128.



In [ ]:
BATCH_SIZE = 128
INDEX_COUNT = min(10_000, len(articles))  # Upsert 10,000 top articles
subset_articles = articles[:INDEX_COUNT]

print(f"[*] Upserting {INDEX_COUNT:,} documents to Qdrant...")

for i in tqdm(range(0, INDEX_COUNT, BATCH_SIZE)):
    batch = subset_articles[i : i + BATCH_SIZE]
    texts = [item["text"] for item in batch]
    vectors = embedder.encode(texts, show_progress_bar=False, normalize_embeddings=True).tolist()
    
    points = [
        models.PointStruct(
            id=item["doc_id"],
            vector=vec,
            payload={
                "brand": item["brand"],
                "category": item["category"],
                "query": item["query"],
                "resolution": item["resolution"]
            }
        )
        for item, vec in zip(batch, vectors)
    ]
    
    client.upsert(
        collection_name=COLLECTION_NAME,
        points=points
    )

print(f"[OK] Successfully indexed {INDEX_COUNT:,} vectors.")



### Step 7: Verify Semantic Search Performance
Test retrieval with sample customer inquiries.



In [ ]:
def search_kb(query: str, brand: str = None, top_k: int = 3):
    query_vec = embedder.encode(query, normalize_embeddings=True).tolist()
    
    query_filter = None
    if brand:
        query_filter = models.Filter(
            must=[
                models.FieldCondition(
                    key="brand",
                    match=models.MatchValue(value=brand)
                )
            ]
        )
    
    # Cross-version compatibility (legacy search vs modern query_points)
    if hasattr(client, 'query_points'):
        res = client.query_points(
            collection_name=COLLECTION_NAME,
            query=query_vec,
            query_filter=query_filter,
            limit=top_k
        )
        return res.points
    elif hasattr(client, 'search'):
        return client.search(
            collection_name=COLLECTION_NAME,
            query_vector=query_vec,
            query_filter=query_filter,
            limit=top_k
        )
    else:
        raise AttributeError('QdrantClient has neither query_points nor search method')

test_queries = [
    ("How do I cancel my subscription and get a refund?", "AmazonHelp"),
    ("My iPhone screen is unresponsive after the latest update", "AppleSupport"),
    ("My flight was delayed and luggage is missing", "Delta")
]

for q, b in test_queries:
    print(f"\n==================================================")
    print(f"Query: '{q}' [Brand Filter: {b}]")
    hits = search_kb(q, brand=b, top_k=2)
    for idx, hit in enumerate(hits, 1):
        print(f"  Hit #{idx} (Score: {hit.score:.4f}):")
        print(f"    Resolution: {hit.payload['resolution']}")



### Step 8: Export Offline Sample KB for Local Development
Export 100 representative entries to `sample_kb.json` to allow developers to run the backend and frontend locally without any cloud setup.



In [ ]:
import json

sample_100 = subset_articles[:100]
offline_export_path = "/content/sample_kb.json"

with open(offline_export_path, "w", encoding="utf-8") as f:
    json.dump(sample_100, f, indent=2, ensure_ascii=False)

print(f"[OK] Exported 100 sample documents to: {offline_export_path}")
print(f"     Copy this file to `scripts/data/sample_kb.json` in your local repo.")

